# Relationformer per P&ID — riproduzione approssimata

Questo notebook implementa la pipeline Relationformer del paper locale 2411.13929v3.pdf usando Dataset-P&ID (paper 2109.03794v1) al posto dei 2.000 sintetici originali non disponibili. È una sostituzione approssimata: sono 500 disegni e mancano esempi tank, pump e inlet/outlet.

Per il fine tuning occorrono ancora **60 P&ID reali annotati**, forniti separatamente. Non usare OPEN100 per sostituirli. Dataset PID entra nel training e non è più un benchmark indipendente; **OPEN100 e PID2Graph Synthetic rimangono esclusi dal training e dalla validation.**

Abilita due GPU T4 nelle impostazioni Kaggle. Il notebook le usa con DataParallel. Abilita Internet per la prima installazione e i pesi ImageNet. Carica come Kaggle Dataset sia il codice aggiornato sia PID2Graph, includendo models/ops; non serve compilare CUDA. Esegui le celle dall'alto verso il basso dopo aver impostato i percorsi.

I default pubblicati sono: input 512×512, batch effettivo 20, 80 epoche, LR 1e-4 / backbone 3e-5, ResNet-101, 400 object token + 1 relation token e loss 2/2/1/4/3. AdamW, ImageNet, scheduler, patience e soglie sono scelte implementative documentate nel README; non tutti sono specificati dal paper.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess
from IPython.display import display, FileLink

# Cerca il codice nei Dataset Kaggle montati e, se necessario, in /kaggle/working.
search_roots = [Path("/kaggle/input"), Path("/kaggle/working"), Path.cwd()]
candidates = sorted({p.parent for root in search_roots if root.exists()
                     for p in root.rglob("train.py")
                     if (p.parent / "models/relationformer_2D.py").is_file()})
assert candidates, f"Codice non trovato. Dataset disponibili: {list(Path('/kaggle/input').iterdir())}"
if len(candidates) > 1: print("Copie di codice trovate; uso la prima:", candidates)
REPO_SOURCE = candidates[0]
REPO = Path("/kaggle/working/relationformer")
if not (REPO / "train.py").is_file():
    shutil.copytree(REPO_SOURCE, REPO,
        ignore=shutil.ignore_patterns(".git", ".venv", "__pycache__", "PID2Graph", "PID2Graph.zip", "data", "trained_weights"))
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["MPLCONFIGDIR"] = "/kaggle/working/matplotlib"
os.environ["RELATIONFORMER_CACHE_DIR"] = "/kaggle/working/pid-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch
assert torch.cuda.is_available(), "Abilita una GPU nelle impostazioni del notebook"
print("Torch:", torch.__version__, "CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0), "memoria libera/totale:", torch.cuda.mem_get_info(0))
print("Spazio disco:", shutil.disk_usage("/kaggle/working"))
def run(*args):
    subprocess.run([sys.executable, *map(str, args)], check=True)


## Dataset-P&ID

L'archivio YOLO contiene solo box e classi dei simboli. Usiamo la copia già convertita in `PID2Graph/Patched/Dataset PID`, che aggiunge nodi strutturali e archi GraphML richiesti dal Relationformer.

Non usare `train (2).txt` e `val (1).txt` dell'archivio: dividono casualmente le patch e quasi tutti i disegni compaiono in entrambi. Il loader crea uno split 95:5 per disegno originale.

Imposta PID2GRAPH alla directory che contiene le sottocartelle `Patched` e `Complete`.


In [ ]:
PID2GRAPH = Path("/kaggle/input/your-pid2graph/PID2Graph")
TRAIN_ROOT = PID2GRAPH / "Patched"
SOURCE = TRAIN_ROOT / "Dataset PID"
assert SOURCE.is_dir(), "Imposta PID2GRAPH: manca Patched/Dataset PID"
drawings = [p for p in SOURCE.iterdir() if p.is_dir() and any(p.glob("*.graphml"))]
assert len(drawings) == 500, f"Attesi 500 disegni Dataset-P&ID: trovati {len(drawings)}"
print("Disegni:", len(drawings), "patch GraphML:", len(list(SOURCE.rglob("*.graphml"))))


In [ ]:
import yaml
config = yaml.safe_load((REPO / "configs/road_2D.yaml").read_text())
config["DATA"]["BATCH_SIZE"] = 2  # se OOM: 1; batch effettivo rimane 20
config["DATA"]["NUM_WORKERS"] = 2
config["DATA"]["CACHE_IMAGES"] = False  # evita memmap >20 GB: lettura lazy
config["TRAIN"]["EFFECTIVE_BATCH_SIZE"] = 20
config["TRAIN"]["MAX_HOURS"] = 9.5  # budget di questa sessione, non completamento del training
config["TRAIN"]["EARLY_STOPPING_PATIENCE"] = 10  # assunzione, attiva solo in fine tuning
CONFIG = REPO / "kaggle_config.yaml"
CONFIG.write_text(yaml.safe_dump(config))


## Controllo dei dati

L'indice valida GraphML, file abbinati, classi e limite di query. Tutte le patch dello stesso disegno condividono lo split 95:5. I box fuori bordo vengono clippati; i grafi senza archi sono ammessi. Il report delle statistiche viene salvato nella cache.


In [ ]:
from dataset_road_network import PatchedPIDDataset
from pid_graph import parse_graph, overlay
dataset = PatchedPIDDataset(TRAIN_ROOT, split="train", train_sources=["Dataset PID"],
    cache_dir=os.environ["RELATIONFORMER_CACHE_DIR"])
image, graph, sample_id = dataset.samples[0]
nodes, edges = parse_graph(graph)
preview = REPO / "sample_annotations.png"
overlay(image, nodes, edges, preview)
from PIL import Image
display(Image.open(preview))
print(sample_id, len(nodes), "nodi,", len(edges), "archi")


## Pretraining e ripresa

Il pretraining parte da backbone ImageNet e teste nuove. L'accumulo di gradienti mantiene batch effettivo 20; l'ultimo gruppo dell'epoca può essere più piccolo. Il run salva last.pt durante l'epoca e best.pt dopo la validation. Un limite di sessione può intervenire prima della prima validation: è normale avere solo last.pt.

Per riprendere, imposta RESUME al last.pt della sessione precedente e usa gli stessi dati/configurazione. Carica l'intera cartella del run, inclusa history.jsonl, per conservare lo storico. Non cambiare microbatch durante un resume.


In [ ]:
OUT = Path("/kaggle/working/pretrain")
RESUME = None  # esempio: Path("/kaggle/input/previous-pretrain/last.pt")
if RESUME is not None and not OUT.exists():
    shutil.copytree(RESUME.parent, OUT)
    RESUME = OUT / RESUME.name
command = ["train.py", "--phase", "pretrain", "--config", CONFIG,
    "--data-root", TRAIN_ROOT, "--synthetic-source", "Dataset PID",
    "--output-dir", OUT, "--device", "cuda", "--cuda_visible_device", 0, 1]
if RESUME is not None: command += ["--resume", RESUME]
run(*command)


In [ ]:
import matplotlib.pyplot as plt
history_path = OUT / "history.jsonl"
if history_path.exists():
    rows = [json.loads(line) for line in history_path.read_text().splitlines() if line]
    plt.plot([r["epoch"] for r in rows], [r["train_loss"]["total"] for r in rows], label="train")
    plt.plot([r["epoch"] for r in rows], [r["validation"]["loss"]["total"] for r in rows], label="validation")
    plt.xlabel("Epoca"); plt.ylabel("Loss totale"); plt.legend(); plt.show()
    previews = sorted((OUT / "validation").glob("*/prediction.png"))
    if previews:
        from PIL import Image
        display(Image.open(previews[-1]))
        display(Image.open(previews[-1].with_name("truth.png")))
else:
    print("Nessuna epoca completa: usa last.pt per continuare; la validation viene eseguita a fine epoca.")


## Conservare checkpoint e dati

Un pretraining completo può richiedere più sessioni. Scarica il run e mantieni invariati Dataset-P&ID, config e split durante il resume. Le prestazioni si misurano dopo il training: il superamento del test di pipeline non dimostra la qualità del modello.


In [ ]:
# Save Version / Run All conserva /kaggle/working negli Output del notebook.
# Puoi scaricare i singoli file da Output, oppure questo archivio del run.
if (OUT / "last.pt").is_file():
    archive = shutil.make_archive(str(OUT) + "_run", "zip", OUT)
    display(FileLink(archive))
    print("Conserva checkpoint, config.yaml e split.json; riusa lo stesso Dataset-P&ID.")
